In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom
from jax_sbgeom.flux_surfaces import ToroidalExtent
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

In [2]:
#flux_surface     =  jax_sbgeom.flux_surfaces.FluxSurface.from_hdf5("/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4")
#s_mesh           = jnp.linspace(0, 1, 10)
#tetra_mesh       =  jax_sbgeom.flux_surfaces.flux_surface_meshing.mesh_tetrahedra(flux_surface, s_mesh, ToroidalExtent.half_module(flux_surface), 50,60)
#face_template    = jsn.tools.mesh_tools._face_template(tetra_mesh[1].shape[1])
#geometric_domain = jsn.domain.GeometricDomain.from_mesh(tetra_mesh, face_template)
#quadrature_set   = jsn.quadrature_set.tn.create_tn_quadrature_set(4)
#pol_mesh         = jax_sbgeom.flux_surfaces.flux_surface_meshing.mesh_poloidal_plane(flux_surface, jnp.linspace(0,1,10)**2, 0.52,  20)
#tri_face_template = jsn.tools.mesh_tools._face_template(pol_mesh[1].shape[1])
#pol_domain        = jsn.domain.GeometricDomain.from_mesh(pol_mesh, tri_face_template)


In [3]:
from finite_elements.test_finite_elements import _get_element_jsn

In [4]:

from basix import ElementFamily, LagrangeVariant
import basix
from jax_sn.solution_domain.basix_fem import BasixLagrangianSimplex, _create_face_quadrature_on_face, _map_simplex_point_to_different_permutation, _map_face_points_to_reference_cell
from jax_sn.tools.permutation_tools import _obtain_permutation, _indices_to_permutation_number
degree = 3
lasimp = BasixLagrangianSimplex(1,3)
lasimp.n_vertices_per_face(0)

3

In [5]:
import StellBlanket as SB

def create_sb_sim():
    materials = SB.Generic_Elements.Material(SB.Generic_Elements.Vacuum_Cross_Section())
    cartesian_tet_domain = SB.Tetrahedrons.Tetrahedron_Create_Slab([1.0],[5], [materials], 1.0, 1)

    verts = cartesian_tet_domain.vertices
    conn = cartesian_tet_domain.connectivity
    face_conn = jsn.tools.mesh_tools._face_template(conn.shape[1])


    quadrature_set = SB.Generic_Elements.TN_Quadrature_Set(4)


    quad_set = SB.Generic_Elements.Quadrature_Set(quadrature_set.Directions()[0:1], quadrature_set.Weights()[0:1])
    energy_set =SB.Generic_Elements.Energy_Set([0,1])


    matfreente = SB.Generic_Elements.Create_RAM_Simulation(quad_set, energy_set, cartesian_tet_domain, 1, 0)
    return matfreente

def sb_sim_to_jsn(sb_sim):
    verts = jnp.array(sb_sim.geometric_domain.vertices)
    conn  = jnp.array(sb_sim.geometric_domain.connectivity)
    face_conn = jsn.tools.mesh_tools._face_template(conn.shape[1])
    geometric_domain = jsn.domain.GeometricDomain.from_mesh((verts, conn), face_conn)
    quad_set = sb_sim.quadrature_set
    quad_set = jsn.quadrature_set.QuadratureSet(jnp.array(quad_set.Directions()), jnp.array(quad_set.Weights()))   
    return geometric_domain, quad_set

sb_sim = create_sb_sim()
order_old = sb_sim.simulation.Order(sb_sim.quadrature_set[0])

Triangle Elements not compiled


W1202 17:55:04.167600   52238 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W1202 17:55:04.170683   52186 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


In [6]:
sb_sim.geometric_domain[0]

Tetrahedron_Element(nodes:[0, -0.5, -0.5], [0.2, -0.5, -0.5], [0, 0.5, -0.5], [0, -0.5, 0.5])

In [7]:
from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
domain, quad_set = sb_sim_to_jsn(sb_sim)


toposort = TopologicalSortedAngle.from_domain_angle(domain, quad_set[0][0])

In [8]:
from jax_sn.solution_domain.solution_domain import T_inv_sweep, FEMMatrices
simplex_deg = BasixLagrangianSimplex(1, domain.connectivity.shape[1] - 1)
femmatrices = FEMMatrices.from_element(simplex_deg)

In [9]:
precomputed_data = simplex_deg.precompute_geometric_data(domain)

In [10]:
from jax_sn.domain.geometric_domain import CrossSectionData

cross_sections_data = CrossSectionData(jnp.ones(domain.n_elements))
source              = jsn.source.Source(jnp.ones((domain.n_elements, simplex_deg.n_basis)))



In [17]:

result = jnp.zeros( (domain.n_elements, simplex_deg.n_basis ))



In [27]:
%timeit T_inv_sweep(femmatrices, domain, toposort, cross_sections_data, precomputed_data, source, result).block_until_ready()

772 μs ± 64 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [28]:
Gijinv = T_inv_sweep(femmatrices, domain, toposort, cross_sections_data, precomputed_data, source, result)

In [29]:
Gijinv

Array([[[ 24.1922695 , -23.99012289, -19.2218024 , -66.90500729],
        [-22.4114402 ,  48.91147977, -17.32738318, -27.49549723],
        [ 51.59143309,   1.88704852,  56.71662673, 113.72191381],
        [-68.87836065, -31.04753328, -21.58982643, -48.91122051]],

       [[-38.27827066, -23.7778485 , -18.56317997, -18.56317997],
        [-21.73136448,  47.26070878, -16.10739915, -16.10739915],
        [ 10.3752403 , -13.53131637,  48.59451895, -15.12508682],
        [ 32.67149362, -11.49827428, -14.44292548,  49.27668029]],

       [[-36.75208036, -70.80678695, -21.00533531, -21.00533531],
        [-69.42130858,  17.8657847 , -19.34276127, -19.34276127],
        [ 18.48590866,   5.0447243 ,  52.13862211, -15.11705224],
        [ 61.71896632,  31.81584592, -13.03883468,  54.21683966]],

       [[ 29.91669107, -18.69669719, -59.70972657, -18.69669719],
        [ 16.48847148,  53.4966074 ,  52.7855176 , -13.75906695],
        [-61.56133791, -20.9186308 , -43.0769121 , -20.9186308 ],
    